
# 03 — Data Cleaning

## Project: ThreatTrace

### Objective
Clean the merged network traffic dataset and prepare it for subsequent model development.

### Cleaning Tasks
1. Load the merged dataset generated in Notebook 2.
2. Inspect dataset dimensions, column names, data types, and target labels.
3. Identify missing, infinite, and duplicate values.
4. Handle invalid values and missing data appropriately.
5. Validate the cleaned dataset.
6. Save the cleaned dataset for the next stage.

### Scope
This notebook focuses only on data cleaning. Feature selection, feature engineering, feature scaling, and model training will be performed separately.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path
import gc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:

from pathlib import Path
import pandas as pd

# Set the project root directory
PROJECT_ROOT = Path.cwd()

# Handle execution from either the project root or notebooks folder
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cicids2017_merged.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATASET_PATH)
print("File exists:", DATASET_PATH.exists())

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

df = pd.read_csv(DATASET_PATH, low_memory=False)

print("\nDataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")


Project root: d:\ThreatTrace
Dataset path: d:\ThreatTrace\data\processed\cicids2017_merged.csv
File exists: True

Dataset loaded successfully.
Dataset shape: (2830743, 79)
Total rows: 2,830,743
Total columns: 79


In [3]:

# Inspect the dataset structure

print("First 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.value_counts().rename_axis("Data Type").to_frame("Column Count"))

print("\nDataset memory usage:")
print(f"{df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")


First 5 rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,33,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,1.100917e+05,18348.62385,109.0,0.0,109,109,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,9174.311927,9174.311927,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,2.307692e+05,38461.53846,52.0,0.0,52,52,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,19230.769230,19230.769230,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,3.529412e+05,58823.52941,34.0,0.0,34,34,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,29411.764710,29411.764710,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,31,329,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN



Column names:
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Coun

,Column Count
Data Type,
int64,54
float64,24
str,1



Dataset memory usage:
1.79 GB


In [4]:

# Remove leading and trailing whitespace from column names

df.columns = df.columns.str.strip()

print("Column names cleaned successfully.")
print("\nFirst 10 column names:")
print(df.columns[:10].tolist())

print("\nTarget column exists:", "Label" in df.columns)


Column names cleaned successfully.

First 10 column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']

Target column exists: True


In [5]:
# Inspect data quality before cleaning

df.columns = df.columns.str.strip()

if "Label" not in df.columns:
    raise ValueError("Target column 'Label' was not found.")

numeric_df = df.select_dtypes(include=[np.number])

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Missing values: {df.isna().sum().sum():,}")
print(f"Infinite numeric values: {np.isinf(numeric_df).sum().sum():,}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")

print("\nColumns containing missing values:")
missing_by_column = df.isna().sum()
display(
    missing_by_column[missing_by_column > 0]
    .sort_values(ascending=False)
    .to_frame("Missing Count")
)

print("\nTarget distribution:")
display(df["Label"].value_counts(dropna=False).to_frame("Count"))

Rows: 2,830,743
Columns: 79
Missing values: 1,358
Infinite numeric values: 4,376
Exact duplicate rows: 308,381

Columns containing missing values:


,Missing Count
Flow Bytes/s,1358



Target distribution:


,Count
Label,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


In [6]:

# Step 1: Create a separate working copy
df_clean = df.copy()

rows_initial = len(df_clean)

# Step 2: Replace infinite numeric values with missing values
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

df_clean[numeric_cols] = df_clean[numeric_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Initial rows:", f"{rows_initial:,}")
print("Missing values after replacing infinity:",
      f"{df_clean.isna().sum().sum():,}")

# Step 3: Remove rows containing missing values
rows_before_missing = len(df_clean)

df_clean = df_clean.dropna().copy()

rows_removed_missing = rows_before_missing - len(df_clean)

print("\nMissing-value cleaning completed.")
print("Rows removed:", f"{rows_removed_missing:,}")
print("Rows remaining:", f"{len(df_clean):,}")

# Step 4: Identify features that must not contain negative values
nonnegative_features = [
    "Flow Duration",
    "Flow Packets/s",
    "Flow Bytes/s",
    "Flow IAT Min",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Fwd IAT Min",
    "Fwd Header Length",
    "Bwd Header Length",
    "min_seg_size_forward"
]

# Ensure the expected features exist
missing_features = [
    col for col in nonnegative_features
    if col not in df_clean.columns
]

if missing_features:
    raise ValueError(
        f"Expected features not found: {missing_features}"
    )

# Step 5: Find rows with at least one negative value
negative_mask = (
    df_clean[nonnegative_features] < 0
).any(axis=1)

rows_before_negative_cleaning = len(df_clean)
rows_removed_negative = int(negative_mask.sum())

print("\nInvalid negative-value audit:")
print("Rows containing invalid negative measurements:",
      f"{rows_removed_negative:,}")

print("\nTarget labels of affected rows:")
display(
    df_clean.loc[negative_mask, "Label"]
    .value_counts()
    .to_frame("Affected records")
)

# Step 6: Remove affected rows
df_clean = df_clean.loc[~negative_mask].copy()

print("\nNegative-value cleaning completed.")
print("Rows removed:", f"{rows_removed_negative:,}")
print("Rows remaining:", f"{len(df_clean):,}")

# Step 7: Report the total effect of cleaning
print("\nCleaning summary:")
print("Original rows:", f"{rows_initial:,}")
print("Final rows before duplicate removal:", f"{len(df_clean):,}")
print("Rows removed for missing values:",
      f"{rows_removed_missing:,}")
print("Rows removed for invalid negative values:",
      f"{rows_removed_negative:,}")
print("Remaining missing values:",
      f"{df_clean.isna().sum().sum():,}")


Initial rows: 2,830,743
Missing values after replacing infinity: 5,734

Missing-value cleaning completed.
Rows removed: 2,867
Rows remaining: 2,827,876

Invalid negative-value audit:
Rows containing invalid negative measurements: 2,925

Target labels of affected rows:


,Affected records
Label,
BENIGN,2731
DoS Hulk,159
DDoS,19
DoS GoldenEye,5
FTP-Patator,4
Heartbleed,4
SSH-Patator,2
Infiltration,1



Negative-value cleaning completed.
Rows removed: 2,925
Rows remaining: 2,824,951

Cleaning summary:
Original rows: 2,830,743
Final rows before duplicate removal: 2,824,951
Rows removed for missing values: 2,867
Rows removed for invalid negative values: 2,925
Remaining missing values: 0


In [7]:
# Examine exact duplicate records

print("Rows before duplicate removal:", f"{len(df_clean):,}")
print("Exact duplicate rows:", f"{df_clean.duplicated().sum():,}")

print("\nTarget distribution before duplicate removal:")
display(df_clean["Label"].value_counts().to_frame("Count"))

Rows before duplicate removal: 2,824,951
Exact duplicate rows: 307,068

Target distribution before duplicate removal:


,Count
Label,
BENIGN,2268589
DoS Hulk,229965
PortScan,158804
DDoS,128006
DoS GoldenEye,10288
FTP-Patator,7931
SSH-Patator,5895
DoS slowloris,5796
DoS Slowhttptest,5499


In [8]:

# Remove exact duplicate records while retaining the first occurrence

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(keep="first").copy()

rows_removed = rows_before - len(df_clean)

print("Duplicate removal completed.")
print(f"Rows before: {rows_before:,}")
print(f"Duplicate rows removed: {rows_removed:,}")
print(f"Rows after: {len(df_clean):,}")
print(f"Remaining exact duplicates: {df_clean.duplicated().sum():,}")


Duplicate removal completed.
Rows before: 2,824,951
Duplicate rows removed: 307,068
Rows after: 2,517,883
Remaining exact duplicates: 0


In [9]:

# Final data-quality checks

numeric_df = df_clean.select_dtypes(include=[np.number])

print("Final dataset shape:", df_clean.shape)
print("Missing values:", int(df_clean.isna().sum().sum()))
print("Infinite numeric values:", int(np.isinf(numeric_df).sum().sum()))
print("Exact duplicate rows:", int(df_clean.duplicated().sum()))
print("Missing target labels:", int(df_clean["Label"].isna().sum()))

print("\nRemaining negative values in validated features:")
display(
    (df_clean[nonnegative_features] < 0)
    .sum()
    .to_frame("Count")
)

print("\nClass distribution:")
display(
    df_clean["Label"]
    .value_counts(dropna=False)
    .rename_axis("Label")
    .to_frame("Count")
)

print("\nData types:")
display(
    df_clean.dtypes.value_counts()
    .rename_axis("Data Type")
    .to_frame("Count")
)

# Stop if any required validation fails
assert df_clean.isna().sum().sum() == 0
assert np.isinf(numeric_df).sum().sum() == 0
assert df_clean.duplicated().sum() == 0
assert df_clean["Label"].notna().all()
assert (df_clean[nonnegative_features] >= 0).all().all()

print("\nAll final data-quality checks passed.")


Final dataset shape: (2517883, 79)
Missing values: 0
Infinite numeric values: 0
Exact duplicate rows: 0
Missing target labels: 0

Remaining negative values in validated features:


,Count
Flow Duration,0
Flow Packets/s,0
Flow Bytes/s,0
Flow IAT Min,0
Flow IAT Mean,0
Flow IAT Max,0
Fwd IAT Min,0
Fwd Header Length,0
Bwd Header Length,0
min_seg_size_forward,0



Class distribution:


,Count
Label,
BENIGN,2092335
DoS Hulk,172688
DDoS,127995
PortScan,90694
DoS GoldenEye,10281
FTP-Patator,5927
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3217



Data types:


,Count
Data Type,
int64,54
float64,24
str,1



All final data-quality checks passed.


In [10]:

# Save the cleaned dataset

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cicids2017_cleaned.csv"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(OUTPUT_PATH, index=False)

print("Cleaned dataset saved successfully.")
print("Output path:", OUTPUT_PATH)
print("Rows:", f"{len(df_clean):,}")
print("Columns:", df_clean.shape[1])
print(
    "File size:",
    f"{OUTPUT_PATH.stat().st_size / (1024**3):.2f} GB"
)


Cleaned dataset saved successfully.
Output path: d:\ThreatTrace\data\processed\cicids2017_cleaned.csv
Rows: 2,517,883
Columns: 79
File size: 0.82 GB


In [11]:

# Verify the saved dataset can be read correctly

df_check = pd.read_csv(OUTPUT_PATH, low_memory=False)

numeric_check = df_check.select_dtypes(include=[np.number])

print("Saved dataset verification")
print("-" * 40)
print(f"Shape: {df_check.shape}")
print(f"Missing values: {df_check.isna().sum().sum():,}")
print(f"Infinite numeric values: {np.isinf(numeric_check).sum().sum():,}")
print(f"Duplicate rows: {df_check.duplicated().sum():,}")
print(f"Missing labels: {df_check['Label'].isna().sum():,}")
print(f"Unique classes: {df_check['Label'].nunique()}")

assert df_check.shape == df_clean.shape
assert df_check.isna().sum().sum() == 0
assert np.isinf(numeric_check).sum().sum() == 0
assert df_check.duplicated().sum() == 0
assert df_check["Label"].notna().all()

print("\nAll saved-dataset checks passed.")
assert (df_check[nonnegative_features] >= 0).all().all()

print("Invalid negative-value check passed.")

Saved dataset verification
----------------------------------------
Shape: (2517883, 79)
Missing values: 0
Infinite numeric values: 0
Duplicate rows: 0
Missing labels: 0
Unique classes: 15

All saved-dataset checks passed.
Invalid negative-value check passed.



## Conclusion

The CICIDS2017 merged dataset was cleaned and validated.

### Cleaning operations
- Standardized column names by removing leading and trailing spaces.
- Replaced infinite numeric values with missing values.
- Removed records containing missing values.
- Removed records containing negative values in validated flow duration, flow rate, inter-arrival-time, header-length, and segment-size features.
- Removed exact duplicate rows while retaining the first occurrence.
- Preserved target labels for the remaining records.

### Final validation
The final row count and class distribution are obtained from the executed notebook output. The dataset is checked for missing values, infinite values, invalid negative measurements, duplicate records, and missing target labels.

### Output
`data/processed/cicids2017_cleaned.csv`

Feature selection, feature transformations, and model training are performed in later notebooks.
